### Imports

In [34]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import nltk
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from collections import Counter
from tqdm import tqdm
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
import time
import os

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder

### Data

In [2]:
def preprocess(df, word2idx=None, label2idx=None, max_len=500, tokenize_func=None):
    texts = df["text"].values
    labels = df["category"].values if "category" in df.columns else None #Check if there is a label, if not, fill in "None".

    # Label encoding.
    if label2idx is None and labels is not None:
        unique_labels = set(labels)
        label2idx = {label: idx for idx, label in enumerate(unique_labels)}
    if labels is not None:
        labels = [label2idx[label] for label in labels]

    # Tokenization.
    tokenized_texts = [word_tokenize(text.lower()) for text in texts]

    if word2idx is None:
        # Build a vocabulary list.
        all_tokens = [token for text in tokenized_texts for token in text]
        vocab = Counter(all_tokens)
        print(len(all_tokens))
        vocab_size = 25000  #The total will not exceed 25,000 words.
        vocab = vocab.most_common(vocab_size - 2)
        word2idx = {word: idx + 2 for idx, (word, _) in enumerate(vocab)}
        word2idx["<unk>"] = 0
        word2idx["<pad>"] = 1

    # Numericalization
    def encode_text(text):
        return [word2idx.get(word, word2idx["<unk>"]) for word in text]

    encoded_texts = [encode_text(text) for text in tokenized_texts]

    # Fill or truncate.
    padded_texts = [
        (
            text[:max_len]
            if len(text) > max_len
            else text + [word2idx["<pad>"]] * (max_len - len(text))
        )
        for text in encoded_texts
    ]

    return padded_texts, labels, word2idx, label2idx

# Define dataset class
class NewsDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = torch.tensor(texts, dtype=torch.long)
        self.labels = torch.tensor(labels, dtype=torch.long) if labels is not None else None

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx] if self.labels is not None else self.texts[idx]
    

def simple_tokenize(text):
    return text.lower().split()


In [3]:
nltk.data.path.append('./train_v2/punkt') #Import word library

# Load the training set of the news dataset.
df = pd.read_csv("./train_v2/train_news.csv")
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['category'])

X_train, y_train, word2idx, label2idx = preprocess(train_df,tokenize_func=simple_tokenize)
X_test, y_test, _, _ = preprocess(test_df,tokenize_func=simple_tokenize, word2idx=word2idx, label2idx=label2idx)

train_dataset = NewsDataset(X_train, y_train)
test_dataset = NewsDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

print(label2idx)
num_labels = len(label2idx)
print("The numbers of different label values can be：",num_labels)

flattened_array = [item for sublist in X_train for item in sublist] # Flatten a two-dimensional array into a one-dimensional array.
unique_elements = set(flattened_array) # Remove duplicate elements.
num_unique_elements = len(unique_elements)
print("The number of different words (including punctuation) in the training set’s features is:", num_unique_elements-1)

336384
{'entertainment': 0, 'business': 1, 'tech': 2, 'sport': 3}
The numbers of different label values can be： 4
The number of different words (including punctuation) in the training set’s features is: 19478


In [4]:
len(train_loader), len(test_loader)

(25, 7)

### Model

In [29]:
# Define a bi-directional LSTM
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes, pad_idx, dropout):
        super(LSTMClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim,padding_idx=pad_idx)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional = True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(2 * hidden_dim, num_classes)
    def forward(self, x):
        embedded = self.embedding(x)
        lstm_output, (hidden, cell) = self.lstm(embedded)
        last_hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        last_hidden = self.dropout(last_hidden)
        logits = self.fc(last_hidden)
        return logits

# Model parameters
embedding_dim = 128
hidden_dim = 128
vocab_size = len(word2idx)
num_classes = 4
pad_idx = word2idx["<pad>"]
dropout = 0.2

In [11]:
def evaluate(model, loader, criterion, device):
    model.eval()
    epoch_loss = 0
    all_preds, all_gt = [], []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            logits = model(inputs)
            loss = criterion(logits, labels)
            epoch_loss+=loss.item()
            preds = logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_gt.extend(labels.cpu().numpy())

    avg_loss = epoch_loss/len(loader)
    res = classification_report(all_gt, all_preds, output_dict=True)
    return avg_loss, res['accuracy'], res['weighted avg']['f1-score']

In [30]:
# Instantiate the model.
model = LSTMClassifier(vocab_size, embedding_dim, hidden_dim, num_classes, pad_idx, dropout)

# Define the loss function and optimizer.
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)

# Train model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = model.to(device)
criterion = criterion.to(device)
num_epochs = 10
history = {"train_loss": [], "train_acc": []}
def categorical_accuracy(preds, y):
    max_preds = preds.argmax(dim=1, keepdim=True)
    correct = max_preds.squeeze(1).eq(y)
    return correct.sum() / y.shape[0]

for epoch in range(num_epochs):
    epoch_loss = 0
    epoch_acc = 0
    model.train()
    for batch in train_loader:
        optimizer.zero_grad()
        inputs, targets = batch
        inputs = inputs.to(device)
        targets = targets.to(device)
        logits = model(inputs)
        loss = criterion(logits, targets)
        acc = categorical_accuracy(logits, targets)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        epoch_acc += acc.item()
    history["train_loss"].append(epoch_loss / len(train_loader))
    history["train_acc"].append(epoch_acc / len(train_loader))
    print(f'Epoch {epoch+1}/{num_epochs}, Train Loss: {epoch_loss/len(train_loader):.4f}, Train Acc: {epoch_acc/len(train_loader):.4f}')

    test_loss, test_acc, test_f1 = evaluate(model, test_loader, criterion, device)
    print(f'Epoch {epoch+1}/{num_epochs}, Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}, Test F1: {test_f1:.4f}')


cuda
Epoch 1/10, Train Loss: 1.3612, Train Acc: 0.3250
Epoch 1/10, Test Loss: 1.3256, Test Acc: 0.4350, Test F1: 0.3714
Epoch 2/10, Train Loss: 1.2157, Train Acc: 0.5575
Epoch 2/10, Test Loss: 1.2153, Test Acc: 0.5400, Test F1: 0.4856
Epoch 3/10, Train Loss: 0.9600, Train Acc: 0.6438
Epoch 3/10, Test Loss: 0.9716, Test Acc: 0.5650, Test F1: 0.5434
Epoch 4/10, Train Loss: 0.5924, Train Acc: 0.8263
Epoch 4/10, Test Loss: 0.7217, Test Acc: 0.7250, Test F1: 0.7088
Epoch 5/10, Train Loss: 0.3195, Train Acc: 0.9137
Epoch 5/10, Test Loss: 0.6199, Test Acc: 0.7450, Test F1: 0.7480
Epoch 6/10, Train Loss: 0.1756, Train Acc: 0.9650
Epoch 6/10, Test Loss: 0.5818, Test Acc: 0.8100, Test F1: 0.8078
Epoch 7/10, Train Loss: 0.0756, Train Acc: 0.9875
Epoch 7/10, Test Loss: 0.6009, Test Acc: 0.7850, Test F1: 0.7831
Epoch 8/10, Train Loss: 0.0460, Train Acc: 0.9925
Epoch 8/10, Test Loss: 0.5912, Test Acc: 0.8050, Test F1: 0.8028
Epoch 9/10, Train Loss: 0.0432, Train Acc: 0.9938
Epoch 9/10, Test Loss: 0.

### Simple

In [38]:
train_texts = train_df['text'].astype('str').fillna('')
train_labels_str = train_df['category']

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train_labels_str)
num_classes = label_encoder.classes_

In [35]:
tfidf_vectorizer = TfidfVectorizer(max_features=2500, stop_words='english', ngram_range=(1,2), min_df=2)
x_train = tfidf_vectorizer.fit_transform(train_texts)

In [40]:
model = LogisticRegression(
    C=1,
    solver='lbfgs',
    penalty='l2',
    max_iter=1000
)
model.fit(x_train, y_train)

test_texts = test_df['text'].astype('str').fillna('')
x_test = tfidf_vectorizer.transform(test_texts)
y_test = test_df['category']

preds_encoded = model.predict(x_test)
preds_str = label_encoder.inverse_transform(preds_encoded)

print(classification_report(y_test, preds_str))

               precision    recall  f1-score   support

     business       0.98      1.00      0.99        58
entertainment       0.95      0.93      0.94        41
        sport       0.97      1.00      0.98        57
         tech       0.98      0.93      0.95        44

     accuracy                           0.97       200
    macro avg       0.97      0.96      0.97       200
 weighted avg       0.97      0.97      0.97       200



c:\Users\raian\source\repos\AI\.env\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
